In [1]:
import os
import sys
os.getcwd()

'C:\\Users\\TWH'

In [2]:
# Auto-install MLflow if missing
try:
    import mlflow
    import mlflow.sklearn
    import mlflow.tensorflow
except ImportError:
    !{sys.executable} -m pip install mlflow
    import mlflow
    import mlflow.sklearn
    import mlflow.tensorflow
    
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

In [3]:
# ==============================================================================
# 1. LOAD DATA & SET UP EXPERIMENT
# ==============================================================================
df = pd.read_csv("data/processed/cleaned_traffic_features.csv")

feature_cols = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "is_weekend",
    "is_holiday",
    "is_severe_weather",
    "is_low_visibility",
    "temp_scaled",
    "clouds_scaled",
]

X = df[feature_cols]
y_reg = df["traffic_volume"]
y_cls = df["high_risk"]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

# Initialize MLflow Experiment
mlflow.set_experiment("Traffic_Volume_and_Risk_Modeling")


2026/09/23 13:41:17 INFO mlflow.tracking.fluent: Experiment with name 'Traffic_Volume_and_Risk_Modeling' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:C:/Users/TWH/mlruns/4', creation_time=1790142077140, effective_trace_archival_retention=None, experiment_id='4', last_update_time=1790142077140, lifecycle_stage='active', name='Traffic_Volume_and_Risk_Modeling', tags={}, trace_location=None, workspace='default'>

In [4]:
# ==============================================================================
# 2. RUN 1: RANDOM FOREST REGRESSOR (Traffic Volume)
# ==============================================================================
with mlflow.start_run(run_name="Random_Forest_Regressor_v1"):
    # Hyperparameters
    params_reg = {
        "n_estimators": 100,
        "max_depth": 14,
        "random_state": 42,
        "n_jobs": -1,
    }

    # Model Training
    model_reg = RandomForestRegressor(**params_reg)
    model_reg.fit(X_train_r, y_train_r)

    # Evaluation
    preds_reg = model_reg.predict(X_test_r)
    mae = mean_absolute_error(y_test_r, preds_reg)
    r2 = r2_score(y_test_r, preds_reg)

    # Log Parameters & Metrics
    mlflow.log_params(params_reg)
    mlflow.log_metric("MAE", mae)
    mlflow.log_metric("R2_Score", r2)
    mlflow.set_tag("model_type", "Regression")
    mlflow.set_tag("target", "traffic_volume")

    # Log Feature Importance Plot as an Artifact
    fig, ax = plt.subplots(figsize=(8, 5))
    pd.Series(model_reg.feature_importances_, index=feature_cols).sort_values().plot(
        kind="barh", ax=ax
    )
    ax.set_title("Feature Importances - Regressor")
    plt.tight_layout()
    plot_path = "reg_feature_importance.png"
    plt.savefig(plot_path)
    plt.close()

    mlflow.log_artifact(plot_path)
    os.remove(plot_path)

    # Log Trained Model Binary
    mlflow.sklearn.log_model(model_reg, artifact_path="model")
    print(
        f"Logged Regressor Run: MAE={mae:.2f}, R2={r2:.4f} under Run ID: {mlflow.active_run().info.run_id}"
    )

2026/09/23 13:41:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.





Logged Regressor Run: MAE=265.38, R2=0.9469 under Run ID: 1bb9c8eb4a084af2b0ddf5c7c827f2f9


In [5]:
# ==============================================================================
# 3. RUN 2: RANDOM FOREST CLASSIFIER (Accident Risk)
# ==============================================================================
with mlflow.start_run(run_name="Random_Forest_Classifier_v1"):
    # Hyperparameters
    params_cls = {
        "n_estimators": 100,
        "max_depth": 12,
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1,
    }

    # Model Training
    model_cls = RandomForestClassifier(**params_cls)
    model_cls.fit(X_train_c, y_train_c)

    # Evaluation
    preds_cls = model_cls.predict(X_test_c)
    probs_cls = model_cls.predict_proba(X_test_c)[:, 1]

    acc = accuracy_score(y_test_c, preds_cls)
    prec = precision_score(y_test_c, preds_cls)
    rec = recall_score(y_test_c, preds_cls)
    f1 = f1_score(y_test_c, preds_cls)
    auc = roc_auc_score(y_test_c, probs_cls)

    # Log Parameters & Metrics
    mlflow.log_params(params_cls)
    mlflow.log_metrics(
        {
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1_Score": f1,
            "ROC_AUC": auc,
        }
    )
    mlflow.set_tag("model_type", "Classification")
    mlflow.set_tag("target", "high_risk")

    # Log Trained Model Binary
    mlflow.sklearn.log_model(model_cls, artifact_path="model")
    print(
        f"Logged Classifier Run: F1={f1:.4f}, AUC={auc:.4f} under Run ID: {mlflow.active_run().info.run_id}"
    )

2026/09/23 13:41:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Logged Classifier Run: F1=0.9376, AUC=0.9976 under Run ID: 75c7a8b8e5a245ad83328aef9bced17d
